# Day 28 Tutorial：v0.3 模板结构检查与交接门槛

> **只检查空白字段模板。** 空单元格不是零，说明行不是实验记录；本 Notebook 不调用任何模型。

## Goal

读取 v0.3 工作簿、验证关键 sheet/字段、统计非空标签，并生成默认“待确认”的字段与建模门槛表。

## Setup

从仓库内自动定位工作簿，不写死个人绝对路径。输出是模板结构事实，不证明已有真实粘合剂数据。

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "adhesive" / "templates").exists():
            return candidate
    raise RuntimeError("请从 ML-practice 仓库内运行。")

REPO_ROOT = find_repo_root()
WORKBOOK = (
    REPO_ROOT / "data" / "adhesive" / "templates"
    / "粘合剂重要化学性质_数据格式_v0.3.xlsx"
)
assert WORKBOOK.exists()
print("Workbook:", WORKBOOK.relative_to(REPO_ROOT))

Workbook: data/adhesive/templates/粘合剂重要化学性质_数据格式_v0.3.xlsx


## Steps

### 1. 读取 sheet 和正式字段

v0.3 的第 4 行是正式字段名，因此使用 `header=3`。

In [2]:
raw_sheets = pd.read_excel(WORKBOOK, sheet_name=None, header=None)
sheet_names = list(raw_sheets)
data = pd.read_excel(
    WORKBOOK,
    sheet_name="01_核心化学性质",
    header=3,
)
print("Sheets:", sheet_names)
print("Rows / columns:", len(data), len(data.columns))

Sheets: ['01_核心化学性质', '02_字段说明', '03_公开依据']
Rows / columns: 0 26


### 2. 检查关键字段和非空标签

即使关键字段齐全或未来出现非空标签，也不能自动越过业务、质量和权限审计。

In [3]:
required_columns = [
    "样品编号",
    "粘合剂化学体系",
    "实测性能名称",
    "实测性能值",
    "实测性能单位",
]
missing_columns = [
    name for name in required_columns if name not in data.columns
]
measured_count = int(data["实测性能值"].notna().sum())
structure_summary = pd.DataFrame([{
    "template_version": "v0.3",
    "sheet_count": len(sheet_names),
    "field_count": len(data.columns),
    "data_row_count": len(data),
    "non_null_measured_value_count": measured_count,
    "missing_required": ", ".join(missing_columns) or "none",
    "interpretation": "structure check only; not modeling approval",
}])
display(structure_summary)

,template_version,sheet_count,field_count,data_row_count,non_null_measured_value_count,missing_required,interpretation
0,v0.3,3,26,0,0,none,structure check only; not modeling approval


### 3. 建立默认待确认的交接表

以下只是空白工作模板，不表示任何门槛已经通过。

In [4]:
modeling_gate = pd.DataFrame([
    ("G01", "首个粘合剂体系已冻结？", "待提供", "化学组", "待确认", True),
    ("G02", "一行代表配方/试样/重复中的哪一种？", "待提供", "化学组", "待确认", True),
    ("G03", "首要目标、单位和测试标准一致？", "待提供", "导师/化学组", "待确认", True),
    ("G04", "配方、固化、基材字段含义稳定？", "待提供", "化学组", "待确认", True),
    ("G05", "重复和批次可用于无泄漏分组？", "待提供", "化学组", "待确认", True),
    ("G06", "缺失与异常规则已记录？", "待提供", "算法组/化学组", "待确认", True),
    ("G07", "原始只读版本和处理日志已建立？", "待提供", "数据负责人", "待确认", True),
    ("G08", "建模、共享和公开权限分别确认？", "待提供", "数据所有者", "待确认", True),
], columns=["gate_id", "question", "evidence", "owner", "status", "blocking"])
display(modeling_gate)

,gate_id,question,evidence,owner,status,blocking
0,G01,首个粘合剂体系已冻结？,待提供,化学组,待确认,True
1,G02,一行代表配方/试样/重复中的哪一种？,待提供,化学组,待确认,True
2,G03,首要目标、单位和测试标准一致？,待提供,导师/化学组,待确认,True
3,G04,配方、固化、基材字段含义稳定？,待提供,化学组,待确认,True
4,G05,重复和批次可用于无泄漏分组？,待提供,化学组,待确认,True
5,G06,缺失与异常规则已记录？,待提供,算法组/化学组,待确认,True
6,G07,原始只读版本和处理日志已建立？,待提供,数据负责人,待确认,True
7,G08,建模、共享和公开权限分别确认？,待提供,数据所有者,待确认,True


## Checks

断言只覆盖 v0.3 预期结构与安全默认值；不会把空表变成训练数据。

In [5]:
assert "01_核心化学性质" in sheet_names
assert not missing_columns
assert measured_count == 0, (
    "模板出现非空标签；必须人工确认是否为说明/真实获准记录，不能自动训练。"
)
assert modeling_gate["status"].eq("待确认").all()
assert modeling_gate["blocking"].all()
print("Structure checks passed. Modeling gate remains CLOSED.")

Structure checks passed. Modeling gate remains CLOSED.


## Next Steps

先由导师/化学组确认体系、目标、行定义、分组和权限；再冻结 v1.0，用 3–5 行获准真实样例验接口。样例只验格式，首批数据通过审计后才从传统模型基线开始。